# DTAT399A. deeptrack.backend.core

<a href="https://colab.research.google.com/github/DeepTrackAI/DeepTrack2/blob/develop/tutorials/3-advanced-topics/DTAT399A_backend.core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.

This advanced tutorial introduces the `backend.core.py` module.

## 1. What is `core.py`?

The `core.py` module is DeepTrack2’s foundation for data management and computational graph construction.

It provides the fundamental classes and abstractions that underpin all DeepTrack2 pipelines, enabling flexible, efficient, and traceable computation.

The key roles of `core.py` are:

- **Data Object Abstractions:**
    Defines simple and validated data containers (`DeepTrackDataObject` and `DeepTrackDataDict`) that store, index, and validate arbitrary data. These classes enable hierarchical and multidimensional organization of complex datasets.

- **Computation Graph Nodes:**
    Implements the `DeepTrackNode` class, which represents a node in a computational graph. Each node can compute, store, and cache values, and can express dependencies on other nodes—enabling the creation of highly flexible and efficient processing pipelines.

- **Lazy Evaluation & Caching:**
    Supports on-demand computation and result caching through lazy evaluation. Nodes only compute their value when required, and cache results for future use until dependencies are invalidated.

- **Operator Overloading for Pipelines:**
    Enables intuitive construction of complex computational graphs using standard Python arithmetic and comparison operators (e.g., +, *, <). This makes pipeline composition both expressive and readable.

- **Dependency Tracking & Propagation:**
    Tracks parent-child and dependency relationships among nodes, so that changes or invalidations automatically propagate through the graph—guaranteeing computational consistency.

- **Citation and Provenance:**
    Integrates citation metadata, ensuring proper academic attribution for work that builds upon DeepTrack2’s infrastructure.

## 2. Using Nodes with Parent-Child Dependencies

In DeepTrack2, nodes represent computational units that can be flexibly linked into graphs by defining dependencies. This allows you to build modular, traceable, and efficient pipelines where changes automatically propagate through the graph.

Below we show how to set up parent-child relationships between nodes, store and compute data, and propagate invalidation when the upstream data changes.

In [2]:
from deeptrack.backend.core import DeepTrackNode

### 2.1. Creating Parent and Child Nodes

We create a parent node and a child node whose value is always twice that of its parent.

In [3]:
# Create parent and child nodes
parent = DeepTrackNode(action=lambda: 10)
child = DeepTrackNode(action=lambda _ID=None: parent(_ID) * 2)

### 2.2. Establishing Parent-Child Dependency

We link the parent and child so that the child automatically tracks changes in the parent. In this way,  the parent is updated or invalidated, this relationship ensures that the child is also kept up to date.

In [4]:
# Establish parent-child dependency
parent.add_child(child)

### 2.3. Storing Values and Computing Results

Let’s assign different values to the parent for different data indices (`_ID`).

In [5]:
# Store values in parent node associated to different _IDs
parent.store(15, _ID=(0,))
parent.store(20, _ID=(1,))

### 2.4. Computing and Accessing Child Values

The child node computes its value based on the current value of the parent for each index.

In [6]:
child(_ID=(0,))

30

In [7]:
child(_ID=(1,))

40

**NOTE:** Calling `child(_ID=(0,))` computes the value if needed, and caches it.
On the other hand, calling `child.current_value((0,))` retrieves the currently cached value without recomputing.
Therefore, you can access the last computed value for a specific index using `.current_value(_ID)`.
If the value hasn’t yet been computed or stored, this will raise a `KeyError`.

In [8]:
# Retrieve the cached value without recomputing
child.current_value((0,))

30

### 2.5. Validation and Invalidation

When you invalidate the parent for a particular _ID, the child’s value for that _ID will also be marked as invalid (since it depends on the parent). This ensures that downstream computations are never out of sync.

In [11]:
# Invalidate parent data for a given ID.
parent.invalidate((0,))
parent.is_valid((0,))

False

In [12]:
child.is_valid((0,))

False

### 2.6. Updating and Recomputing Values

After invalidation, if we update the parent and request the child’s value again, it will be recomputed as needed.

In [13]:
# Update the parent value and recompute the child value
parent.store(25, _ID=(0,))
child((0,))

50

In [14]:
parent.is_valid((0,))

True

In [15]:
child.is_valid((0,))

True

### 2.7 Setting a Value and Automatic Invalidation

You can force a value into a node’s storage with `.set_value(value, _ID)`. If the new value is different, dependent nodes will be invalidated.

In [16]:
parent.set_value(100, _ID=(1,))
parent.current_value((1,))

100

In [17]:
parent.is_valid((1,))

True

In [18]:
child.is_valid((1,))

False

## 3. Lazy Evaluation and Caching

A powerful feature of DeepTrack2 nodes is lazy evaluation: the node’s value is only computed when it is needed, and the result is cached until the node (or its dependencies) is invalidated. This avoids redundant computations and ensures high efficiency, especially in large graphs.

In this example, we’ll use a global counter to demonstrate when the node’s computation actually happens.

### 3.1 Defining a Node with a Side Effect

First, we define a calculation function that increments a global counter each time it is called. This allows us to see exactly how many times the node’s computation is performed.

In [19]:
# Create counter node with side effect
call_count = 0
def calculation():
    global call_count
    call_count += 1
    return 10

node = DeepTrackNode(calculation)

### 3.2 Demonstrating Lazy Evaluation

Let’s see what happens when we call the node multiple times:

In [20]:
# First call computes the value (calls the function)
node()

10

In [21]:
call_count

1

In [22]:
# Second call uses the cached value (no additional computation)
node()

10

In [23]:
call_count

1

### 3.3 Invalidation Forces Recalculation

If we invalidate the node, the cache is cleared and the next call will recompute the value:

In [24]:
# Invalidate the node and call again (forces recomputation)
node.invalidate()
node()

10

In [25]:
call_count

2

In [26]:
# Invalidate and call again
node.invalidate()
node()

10

In [27]:
call_count

3

## 4. Data Management with IDs

In DeepTrack2, the `DeepTrackDataDict` class provides an efficient, validated way to manage multiple data objects, each indexed by a unique tuple of integers.

This is especially useful for working with multidimensional datasets, or for mapping results to experiment or batch indices.

### 4.1. Creating and Indexing Data Objects

You can create entries with arbitrary integer index tuples, just like keys in a nested dictionary.

In [28]:
from deeptrack.backend.core import DeepTrackDataDict

data_dict = DeepTrackDataDict()

# Create listings with unique indices.
data_dict.create_index((0, 0))
data_dict.create_index((0, 1))
data_dict.create_index((1, 0))
data_dict.create_index((1, 1))

### 4.2. Storing and Retrieving Data

Each index corresponds to a `DeepTrackDataObject`, where you can store and retrieve data.
This is similar to using a multidimensional dictionary.

In [29]:
# Store some data for the indices.
data_dict[(0, 0)].store("Cat")
data_dict[(0, 1)].store("Dog")
data_dict[(1, 0)].store("Mouse")
data_dict[(1, 1)].store("Bird")

### 4.3. Accessing Data by ID

You can access data by its full index, or get a dictionary of all entries with a common prefix.

In [30]:
# Retrieve and print values for specific indices.
data_dict[(0, 0)].current_value()

'Cat'

In [31]:
data_dict[(1, 1)].current_value()

'Bird'

In [32]:
# Retrieve all entries whose indices start with (0,)
print(data_dict[(0, )])

{(0, 0): <deeptrack.backend.core.DeepTrackDataObject object at 0x307841a50>, (0, 1): <deeptrack.backend.core.DeepTrackDataObject object at 0x307843040>}


## 5. Operator Overloading and Pipeline Composition

A unique and powerful feature of `DeepTrackNode` is its support for operator overloading. This allows you to build complex computational pipelines by composing nodes using familiar arithmetic and comparison operators, making your code both expressive and readable.

Every operator creates a new node that, when called, evaluates its operands, applies the operator, and caches the result. Dependency relationships are automatically tracked, so invalidating an operand will invalidate any composed nodes as well.

### 5.1. Combining Nodes with Arithmetic Operators

You can add, subtract, multiply, or divide nodes just like numbers. The result is always a new `DeepTrackNode` that represents the composed computation.

In [33]:
from deeptrack.backend.core import DeepTrackNode

a = DeepTrackNode(lambda: 5)
b = DeepTrackNode(lambda: 3)

In [34]:
sum_node = a + b
sum_node()

8

In [35]:
diff_node = a - b
diff_node()

2

In [36]:
prod_node = a * 2
prod_node()

10

In [37]:
div_node = a / b
div_node()

1.6666666666666667

In [38]:
floordiv_node = a // b
floordiv_node()

1

### 5.2. Chaining and Nesting Operators

You can compose pipelines of arbitrary depth and complexity:

In [39]:
complex_node = ((a + b) * 2) / (b + 1)
complex_node()

4.0

### 5.3. Comparison Operators for Graphs

Comparison operators also work on nodes, returning new nodes that compute boolean results:

In [40]:
lt_node = a < b
lt_node()

False

In [41]:
ge_node = a >= b
ge_node()

True

### 5.4. Mixing Nodes and Constants

You can mix DeepTrackNode instances and regular numbers:

In [42]:
sum_with_constant = a + 7
sum_with_constant()

12

In [43]:
mult_with_constant = 3 * b
mult_with_constant()

9

## 6. Getting Citations

The `DeepTrackNode` class can also be used to obtain the relevant citations.

In [44]:
DeepTrackNode().get_citations()

{'\n@article{Midtvet2021Quantitative,\n    author  = {Midtvedt, Benjamin and Helgadottir, Saga and Argun, Aykut and \n               Pineda, Jesús and Midtvedt, Daniel and Volpe, Giovanni},\n    title   = {Quantitative digital microscopy with deep learning},\n    journal = {Applied Physics Reviews},\n    volume  = {8},\n    number  = {1},\n    pages   = {011310},\n    year    = {2021},\n    doi     = {10.1063/5.0034891}\n}\n'}